In [2]:
import os
import glob
import tqdm
import torchaudio
import pandas as pd

def get_duration(audio_path):
    info = torchaudio.info(audio_path)
    duration = info.num_frames / info.sample_rate
    return duration

In [3]:
root_dir = '/data/mtseng/voice_datasets/LibriTTS/data'
speaker_folders = glob.glob(os.path.join('./data', '*'))
print(f"Found {len(speaker_folders)} speaker folders.")

Found 375 speaker folders.


In [9]:
out_parallel_data = './parallel_data'
tts_output_dir = os.path.join(out_parallel_data, 'elevenlab_tts')
golden_speakers_output_dir = os.path.join(out_parallel_data, 'golden_speakers')
os.makedirs(tts_output_dir, exist_ok=True)
os.makedirs(golden_speakers_output_dir, exist_ok=True)

for speaker_folder in tqdm.tqdm(speaker_folders):
    wav_files = glob.glob(os.path.join(speaker_folder, 'wav', '*.wav'))
    for wav_file in wav_files:
        original_transcript_file = wav_file.replace('.wav', '.txt').replace('/wav/', '/transcript/')
        original_transcript_file = original_transcript_file.replace('./data/', root_dir + '/')
        original_wav_file = wav_file.replace('./data/', root_dir + '/')\
        
        tts_wav_file_path = wav_file.replace('./data/', './raw_data/')
        out_tts_wav_file_path = wav_file.replace('./data/', tts_output_dir + '/')
        out_original_wav_file_path = wav_file.replace('./data/', golden_speakers_output_dir + '/')
        
        assert os.path.exists(original_transcript_file), f"Transcript file not found for {wav_file}: {original_transcript_file}"
        assert os.path.exists(original_wav_file), f"Wav file not found: {original_wav_file}"

        original_duration = get_duration(original_wav_file)
        wav_duration = get_duration(wav_file)
        assert abs(original_duration - wav_duration) < 0.1, f"Duration mismatch for {wav_file}: original {original_duration:.2f}s vs copied {wav_duration:.2f}s"

        # print(f"Checking for transcript file: {original_transcript_file}")
        transcript_file = wav_file.replace('.wav', '.txt').replace('/wav/', '/transcript/')
        os.makedirs(os.path.dirname(transcript_file), exist_ok=True)
        if not os.path.exists(transcript_file):
            os.system("cp {} {}".format(original_transcript_file, transcript_file))
        
        os.makedirs(os.path.dirname(out_tts_wav_file_path), exist_ok=True)
        os.system("cp {} {}".format(tts_wav_file_path, out_tts_wav_file_path))  

        os.makedirs(os.path.dirname(out_original_wav_file_path), exist_ok=True)
        os.system("cp {} {}".format(original_wav_file, out_original_wav_file_path))

  0%|          | 0/375 [00:00<?, ?it/s]/tmp/ipykernel_318151/450469519.py:8: UserWarning: torchaudio._backend.utils.info has been deprecated. This deprecation is part of a large refactoring effort to transition TorchAudio into a maintenance phase. The decoding and encoding capabilities of PyTorch for both audio and video are being consolidated into TorchCodec. Please see https://github.com/pytorch/audio/issues/3902 for more information. It will be removed from the 2.9 release. 
  info = torchaudio.info(audio_path)
/home/mtseng/miniconda3/envs/DarkStream/lib/python3.10/site-packages/torchaudio/_backend/sox.py:24: UserWarning: torchaudio._backend.common.AudioMetaData has been deprecated. This deprecation is part of a large refactoring effort to transition TorchAudio into a maintenance phase. The decoding and encoding capabilities of PyTorch for both audio and video are being consolidated into TorchCodec. Please see https://github.com/pytorch/audio/issues/3902 for more information. It wil

In [6]:
all_speaker_ids = [int(p) for p in os.listdir('./data')]
df = pd.read_csv(os.path.dirname(root_dir) + '/metadata/speakers.csv')
curated_df = df[df['READER'].isin(all_speaker_ids)].copy()
curated_df['GENDER'] = curated_df['GENDER'].astype(str).str.strip()

In [7]:
metadata_out_path = './metadata/speakers.csv'
os.makedirs(os.path.dirname(metadata_out_path), exist_ok=True)
curated_df.to_csv(metadata_out_path, index=False)